
# SHML — Final Experimental Results and Visualizations

## Research Focus

This notebook analyzes the results of the Self-Healing
Machine Learning (SHML) system using the BILL_AMT1 feature
from the UCI Default of Credit Card Clients dataset.

### Objectives

1. Compare model performance under different missingness levels.
2. Evaluate zero and median imputation strategies.
3. Analyze F1 and ROC-AUC performance tradeoffs.
4. Visualize safety acceptance, fallback, and abstention.
5. Compare validation findings against held-out test results.

All results are loaded from previously saved experiment files.
No models are retrained, and no test-based policy tuning occurs.

The UCI source workbook contains no missing values. The saved results come from deliberate `BILL_AMT1` masking on validation copies and, separately, held-out test copies.


In [1]:

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the SHML project directory
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError(
            "Could not locate the SHML project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Figures directory:", FIGURES_DIR)


Project root: /Users/divya/Desktop/School/IU/GitStuff/SHML
Figures directory: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/figures


In [2]:

final_test = pd.read_csv(
    RESULTS_DIR / "final_test" / "final_test_results.csv"
)

robustness = pd.read_csv(
    RESULTS_DIR / "bill_amt1" /
    "integrated_20_seed_results.csv"
)

comparison = pd.read_csv(
    RESULTS_DIR / "bill_amt1" /
    "integrated_vs_notebook.csv"
)

print("Final test rows:", len(final_test))
print("Robustness experiment rows:", len(robustness))
print("Notebook comparison rows:", len(comparison))

assert len(final_test) == 8
assert len(robustness) == 20
assert len(comparison) == 20
assert comparison["matches"].all()

display(final_test)


Final test rows: 8
Robustness experiment rows: 20
Notebook comparison rows: 20


,missingness,strategy,f1,precision,recall,accuracy,roc_auc,f1_vs_healthy,auc_vs_healthy,posthoc_safety_decision,posthoc_safety_reason
0,0.00,Fixed median,0.348416,0.700000,0.231928,0.808000,0.716954,0.000000,0.000000,ACCEPT,All checks passed
1,0.00,Zero imputation,0.348416,0.700000,0.231928,0.808000,0.716954,0.000000,0.000000,ACCEPT,All checks passed
2,0.05,Fixed median,0.352588,0.697329,0.235944,0.808222,0.716666,0.004172,-0.000288,ACCEPT,All checks passed
3,0.05,Zero imputation,0.354525,0.695015,0.237952,0.808222,0.716450,0.006109,-0.000504,ACCEPT,All checks passed
4,0.15,Fixed median,0.354959,0.689855,0.238956,0.807778,0.715461,0.006543,-0.001493,ACCEPT,All checks passed
5,0.15,Zero imputation,0.363905,0.691011,0.246988,0.808889,0.714970,0.015489,-0.001984,ACCEPT,All checks passed
6,0.30,Fixed median,0.360947,0.685393,0.244980,0.808000,0.712948,0.012530,-0.004006,REJECT,ROC-AUC loss exceeds tolerance
7,0.30,Zero imputation,0.378733,0.689655,0.261044,0.810444,0.712371,0.030316,-0.004582,REJECT,ROC-AUC loss exceeds tolerance


In [4]:

import plotly.graph_objects as go

f1_table = final_test.pivot(
    index="missingness",
    columns="strategy",
    values="f1"
).sort_index()

healthy_f1 = final_test.loc[
    final_test["missingness"] == 0,
    "f1"
].iloc[0]

fig = go.Figure()

for strategy in f1_table.columns:
    fig.add_trace(
        go.Scatter(
            x=f1_table.index * 100,
            y=f1_table[strategy],
            mode="lines+markers",
            name=strategy,
            hovertemplate=(
                "Missingness: %{x:.0f}%<br>"
                "F1: %{y:.6f}"
                "<extra>" + strategy + "</extra>"
            )
        )
    )

fig.add_hline(
    y=healthy_f1,
    line_dash="dash",
    annotation_text="Healthy baseline",
    annotation_position="top left"
)

fig.update_layout(
    title="SHML: Effect of BILL_AMT1 Missingness on F1",
    xaxis_title="Injected Missingness (%)",
    yaxis_title="F1 Score",
    xaxis=dict(tickvals=[0, 5, 15, 30]),
    template="plotly_white",
    hovermode="x unified",
    height=500,
    legend_title="Recovery Strategy"
)

fig.show()



### Interpretation — F1 Performance

On the held-out test partition, neither imputation
strategy exhibited F1 degradation relative to the
healthy baseline under the evaluated missingness masks.

At 30% missingness, zero imputation achieved an F1
score of 0.378733, compared with the healthy F1
of 0.348416.

Zero imputation also outperformed fixed median imputation
at the evaluated nonzero missingness levels.

These results do not establish successful restoration
from F1 degradation because no such degradation
was observed in this experiment.

The findings motivate evaluating additional performance
metrics and safety constraints.


In [5]:

auc_table = final_test.pivot(
    index="missingness",
    columns="strategy",
    values="roc_auc"
).sort_index()

healthy_auc = final_test.loc[
    final_test["missingness"] == 0,
    "roc_auc"
].iloc[0]

fig_auc = go.Figure()

for strategy in auc_table.columns:
    fig_auc.add_trace(
        go.Scatter(
            x=auc_table.index * 100,
            y=auc_table[strategy],
            mode="lines+markers",
            name=strategy,
            hovertemplate=(
                "Missingness: %{x:.0f}%<br>"
                "ROC-AUC: %{y:.6f}"
                "<extra>" + strategy + "</extra>"
            )
        )
    )

fig_auc.add_hline(
    y=healthy_auc,
    line_dash="dash",
    annotation_text="Healthy ROC-AUC"
)

fig_auc.add_hline(
    y=healthy_auc * (1 - 0.005),
    line_dash="dot",
    line_color="red",
    annotation_text="0.5% Safety Limit",
    annotation_position="bottom left"
)

fig_auc.update_layout(
    title="SHML: ROC-AUC Degradation and Safety Threshold",
    xaxis_title="Injected Missingness (%)",
    yaxis_title="ROC-AUC",
    xaxis=dict(tickvals=[0, 5, 15, 30]),
    template="plotly_white",
    hovermode="x unified",
    height=500,
    legend_title="Recovery Strategy"
)

fig_auc.show()


In [6]:

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig.write_html(
    FIGURES_DIR / "interactive_f1.html",
    include_plotlyjs=True
)

fig_auc.write_html(
    FIGURES_DIR / "interactive_roc_auc.html",
    include_plotlyjs=True
)

print("Interactive plots saved successfully.")


Interactive plots saved successfully.


In [7]:

import plotly.express as px

decision_labels = {
    "PREFERRED_ACCEPTED": "Zero accepted",
    "FALLBACK_ACCEPTED": "Median fallback",
    "NO_SAFE_RECOVERY_ABSTAIN": "Abstained",
}

decision_order = [
    "Zero accepted",
    "Median fallback",
    "Abstained",
]

decision_counts = (
    robustness["status"]
    .map(decision_labels)
    .value_counts()
    .reindex(decision_order, fill_value=0)
    .rename_axis("Decision")
    .reset_index(name="Runs")
)

decision_counts["Percentage"] = (
    decision_counts["Runs"] /
    decision_counts["Runs"].sum() * 100
)

fig_decisions = px.bar(
    decision_counts,
    x="Decision",
    y="Runs",
    color="Decision",
    text="Runs",
    custom_data=["Percentage"],
    color_discrete_map={
        "Zero accepted": "#2A9D8F",
        "Median fallback": "#E9A23B",
        "Abstained": "#D95D5D",
    },
    title="SHML: Recovery Decisions Across 20 Seeds",
)

fig_decisions.update_traces(
    textposition="outside",
    hovertemplate=(
        "%{x}<br>"
        "Runs: %{y}<br>"
        "Percentage: %{customdata[0]:.0f}%"
        "<extra></extra>"
    ),
)

fig_decisions.update_layout(
    template="plotly_white",
    xaxis_title="Recovery Decision",
    yaxis_title="Number of Runs",
    yaxis=dict(range=[0, 20], dtick=2),
    showlegend=False,
    height=500,
)

fig_decisions.show()

fig_decisions.write_html(
    FIGURES_DIR / "recovery_decisions.html",
    include_plotlyjs=True,
)



### Interpretation — Recovery Policy Decisions

At 15% BILL_AMT1 missingness, the integrated SHML
system evaluated 20 reproducible corruption patterns.

- Zero imputation was accepted in 16 cases (80%).
- Median imputation was selected as a fallback in
  2 cases (10%).
- Both strategies were rejected in 2 cases (10%),
  resulting in abstention.

All 20 integrated decisions matched the independent
research notebook results.

These findings demonstrate reproducible, validation-based
safety decisions and the ability to reject recovery
candidates that violate predefined performance constraints.

They do not establish autonomous recovery in an
unlabeled production environment.


In [9]:

import plotly.express as px

seed_data = robustness.copy()

decision_labels = {
    "PREFERRED_ACCEPTED": "Zero accepted",
    "FALLBACK_ACCEPTED": "Median fallback",
    "NO_SAFE_RECOVERY_ABSTAIN": "Abstained",
}

seed_data["decision"] = (
    seed_data["status"].map(decision_labels)
)

seed_data["seed_label"] = (
    seed_data["seed"].astype(str)
)

fig_seeds = px.scatter(
    seed_data,
    x="seed",
    y="decision",
    color="decision",
    hover_data={
        "seed": True,
        "selected_action": True,
        "preferred_reason": True,
        "fallback_reason": True,
        "decision": False,
    },
    category_orders={
        "decision": [
            "Zero accepted",
            "Median fallback",
            "Abstained",
        ]
    },
    color_discrete_map={
        "Zero accepted": "#2A9D8F",
        "Median fallback": "#E9A23B",
        "Abstained": "#D95D5D",
    },
    title="SHML: Recovery Decisions by Random Seed",
)

fig_seeds.update_traces(
    marker=dict(size=15, line=dict(width=1, color="white"))
)

fig_seeds.update_layout(
    template="plotly_white",
    xaxis=dict(
        title="Random Seed",
        tickmode="linear",
        dtick=1,
        range=[-0.5, 19.5],
    ),
    yaxis_title="Recovery Decision",
    legend_title="Decision",
    height=500,
)
fig_seeds.update_xaxes(
    tickmode="linear",
    tick0=0,
    dtick=1,
    tickangle=-45,
    range=[-0.5, 19.5],
)

fig_seeds.update_layout(
    width=1100,
    height=550,
    margin=dict(b=100, r=150),
)

fig_seeds.show()

fig_seeds.write_html(
    FIGURES_DIR / "seed_level_decisions.html",
    include_plotlyjs=True,
)



### Interpretation — Seed-Level Recovery Decisions

The recovery policy was evaluated using 20 distinct
random missingness patterns at 15% missingness.

The policy selected zero imputation for 16 seeds,
median imputation for seeds 0 and 2, and abstained
for seeds 8 and 11.

This demonstrates that recovery decisions can vary
depending on which BILL_AMT1 observations are missing.

The integrated results matched the independent notebook
results for all 20 seeds.

Because all experiments use the same validation partition,
the seeds represent variation in corruption patterns,
not independent samples from the population.


In [13]:

import plotly.express as px

# --------------------------------------------------
# Step 1: Prepare the final test results
# --------------------------------------------------

tradeoff = final_test.copy()

tradeoff["missingness_pct"] = (
    tradeoff["missingness"] * 100
)

tradeoff["safety"] = tradeoff[
    "posthoc_safety_decision"
].map({
    "ACCEPT": "Accepted",
    "REJECT": "Rejected"
})

# --------------------------------------------------
# Step 2: Define the frozen safety threshold
# --------------------------------------------------

healthy_auc = final_test.loc[
    final_test["missingness"] == 0,
    "roc_auc"
].iloc[0]

auc_limit = healthy_auc * (1 - 0.005)

# --------------------------------------------------
# Step 3: Create the interactive scatter plot
# --------------------------------------------------

fig_tradeoff = px.scatter(
    tradeoff,
    x="roc_auc",
    y="f1",
    color="strategy",
    symbol="safety",
    size="missingness_pct",
    size_max=18,
    hover_data={
        "missingness_pct": ":.0f",
        "strategy": True,
        "safety": True,
        "roc_auc": ":.6f",
        "f1": ":.6f"
    },
    color_discrete_map={
        "Fixed median": "#636EFA",
        "Zero imputation": "#EF553B"
    },
    symbol_map={
        "Accepted": "circle",
        "Rejected": "diamond"
    },
    title="SHML: F1 vs ROC-AUC Recovery Tradeoff"
)

# --------------------------------------------------
# Step 4: Add ROC-AUC safety threshold
# --------------------------------------------------

fig_tradeoff.add_vline(
    x=auc_limit,
    line_dash="dash",
    line_color="red",
    line_width=2
)

fig_tradeoff.add_annotation(
    x=auc_limit,
    y=1.06,
    xref="x",
    yref="paper",
    text="ROC-AUC Safety Limit (0.5%)",
    showarrow=False,
    font=dict(size=12, color="red")
)

# --------------------------------------------------
# Step 5: Improve markers
# --------------------------------------------------

fig_tradeoff.update_traces(
    marker=dict(
        sizemin=9,
        line=dict(
            width=1,
            color="white"
        )
    ),
    opacity=0.9
)

# --------------------------------------------------
# Step 6: Improve layout
# --------------------------------------------------

fig_tradeoff.update_layout(
    template="plotly_white",

    width=1050,
    height=600,

    title=dict(
        text="SHML: F1 vs ROC-AUC Recovery Tradeoff",
        x=0.5,
        xanchor="center"
    ),

    xaxis_title="ROC-AUC (Higher Is Better)",
    yaxis_title="F1 Score (Higher Is Better)",

    legend=dict(
        title="Recovery Strategy / Safety",
        x=1.03,
        y=1,
        xanchor="left",
        yanchor="top"
    ),

    margin=dict(
        l=90,
        r=280,
        t=120,
        b=80
    ),

    hovermode="closest"
)

fig_tradeoff.update_xaxes(
    tickformat=".4f",
    showgrid=True,
    gridcolor="#E5ECF6"
)

fig_tradeoff.update_yaxes(
    tickformat=".3f",
    showgrid=True,
    gridcolor="#E5ECF6"
)

# --------------------------------------------------
# Step 7: Display and save
# --------------------------------------------------

fig_tradeoff.show()

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig_tradeoff.write_html(
    FIGURES_DIR / "f1_auc_tradeoff.html",
    include_plotlyjs=True
)

print("Interactive tradeoff visualization saved.")
print("ROC-AUC safety threshold:", round(auc_limit, 6))



Interactive tradeoff visualization saved.
ROC-AUC safety threshold: 0.713369



### Interpretation — Multi-Metric Recovery Safety

The held-out test results reveal a tradeoff between
F1 and ROC-AUC under increasing BILL_AMT1 missingness.

At 30% missingness, zero imputation produced the highest
F1 score (0.378733), compared with the healthy baseline
(0.348416).

However, ROC-AUC decreased from 0.716954 to 0.712371,
exceeding the frozen 0.5% relative-loss tolerance.

The same safety constraint rejected median imputation
at 30% missingness.

These results demonstrate why evaluating recovery using
F1 alone may lead to accepting candidates that violate
other model-performance requirements.

The safety decisions shown here are retrospective audits
using held-out test labels. They do not demonstrate
real-time recovery selection without labels.



### SHML Workflow Interpretation

The SHML system follows a structured recovery process:

1. Inspect incoming data for missing values.
2. Detect whether missingness exceeds the configured threshold.
3. Diagnose the failure severity.
4. Generate a preferred zero-imputation candidate.
5. Evaluate the candidate against data-integrity and
   predictive-performance safety criteria.
6. Accept the preferred candidate if it passes.
7. Otherwise, evaluate training-median imputation.
8. Accept the fallback if it passes, or abstain
   if neither strategy satisfies the safety constraints.

The existing implementation performs predictive-performance
safety evaluation using labeled validation data.

Therefore, this workflow represents an offline experimental
recovery policy, not a fully autonomous production system.
